# Week 37, Day 3: Neuropixels units: quality metrics and a visual hierarchy

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Contamination from violations

Write isi_violation_rate(st, T, tr=0.0015): the number of intervals shorter than tr divided by 2 tr N² / T, with N the number of spikes. Then turn the metric m into a contamination estimate, 1 - sqrt(1 - 2m), clipping m just below 0.5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2011)

def make_unit(rate, T, contam, ref=0.002):
    """A refractory neuron firing at rate (1 - contam), plus Poisson spikes from another source at rate contam."""
    isi = r.exponential(1 / (rate * (1 - contam) + 1e-12), int(rate * T * 1.5)) + ref
    own = np.cumsum(isi); own = own[own < T]
    other = r.uniform(0, T, r.poisson(rate * contam * T))
    return np.sort(np.r_[own, other])

def isi_violation_rate(st, T, tr=0.0015):
    ...

T = 1800.0
est = {}
for c in (0.0, 0.05, 0.1, 0.2, 0.4):
    st = make_unit(10, T, c)
    m = isi_violation_rate(st, T)
    est[c] = 1 - np.sqrt(1 - 2 * min(m, 0.4999))
    print(f"true contamination {c:.2f}: violations {np.mean(np.diff(st) < 0.0015):.4f} of intervals, metric {m:.3f}, estimate {est[c]:.3f}")

In [ ]:
import numpy as np
def test_clean():
    "a clean unit has an estimate of zero"
    assert est[0.0] == 0

def test_order():
    "the inverted estimate rises with true contamination and is within 0.06 of it at every level"
    v = [est[c] for c in sorted(est)]
    assert all(a < b for a, b in zip(v, v[1:]))
    assert all(abs(est[c] - c) < 0.06 for c in est)

def test_metric():
    "the raw metric of a fully random unit is close to 0.5"
    st = np.sort(r.uniform(0, T, int(10 * T)))
    assert abs(isi_violation_rate(st, T) - 0.5) < 0.05

test_clean()
test_order()
test_metric()
print("All checks passed")

### Exercise: Filtering units

Write presence_ratio(st, T): split the session into 60 s blocks and return the fraction of blocks containing at least one spike. Then keep units with estimated contamination below 0.2 and presence ratio above 0.9.

In [ ]:
def presence_ratio(st, T, block=60.0):
    ...

n_units = 80
contam = np.where(r.random(n_units) < 0.25, r.uniform(0.6, 1.0, n_units), r.uniform(0, 0.15, n_units))
lost = np.where(r.random(n_units) < 0.2, r.uniform(0.3, 0.7, n_units), 0.0)     # fraction of session missing
rates = r.lognormal(1.2, 0.8, n_units)
units = []
for i in range(n_units):
    st = make_unit(rates[i], T, contam[i])
    if lost[i] > 0:
        st = st[st < T * (1 - lost[i])]
    units.append(st)
viol = np.array([isi_violation_rate(st, T) for st in units])
fhat = 1 - np.sqrt(1 - 2 * np.minimum(viol, 0.4999))
pres = np.array([presence_ratio(st, T) for st in units])
keep = (fhat < 0.2) & (pres > 0.9)
good = (contam < 0.2) & (lost == 0)
print(f"kept {keep.sum()} of {n_units} units; {keep[good].mean():.0%} of good units kept, {keep[~good].mean():.0%} of bad units kept")

In [ ]:
import numpy as np
def test_presence():
    "presence ratio is 1 for a unit that fires all session and about 0.5 for one present half the time"
    assert presence_ratio(make_unit(5, T, 0), T) == 1.0
    half = make_unit(5, T, 0); half = half[half < T / 2]
    assert 0.45 < presence_ratio(half, T) < 0.55

def test_filter():
    "the filter keeps most good units and removes most bad ones"
    assert keep[good].mean() > 0.9 and keep[~good].mean() < 0.2

test_presence()
test_filter()
print("All checks passed")

### Exercise: Ordering areas by latency

Write latency(rate): with the baseline taken from bins before onset, return the centre of the first bin after onset at which the rate exceeds the baseline mean plus 4 standard deviations in two consecutive bins (nan if never).

In [ ]:
from scipy.stats import spearmanr
areas = ["LGd", "VISp", "VISl", "VISrl", "VISal", "VISpm", "VISam"]
lat_mu = [0.034, 0.045, 0.052, 0.050, 0.056, 0.063, 0.068]          # simulated means, seconds
edges = np.arange(-0.1, 0.25001, 0.002); centres = (edges[:-1] + edges[1:]) / 2

def latency(rate):
    b = rate[centres < 0]
    above = ...
    idx = np.where(above[:-1] & above[1:])[0]
    return centres[idx[0]] if len(idx) else np.nan

area_lat = {}
for a, mu in zip(areas, lat_mu):
    lats = []
    for u in range(10):
        lat = max(0.02, r.normal(mu, 0.006)); base = r.uniform(3, 12); gain = r.uniform(20, 60)
        counts = np.zeros(len(centres))
        for k in range(150):
            sp = list(r.uniform(-0.1, 0.25, r.poisson(base * 0.35))) + list(lat + r.exponential(0.015, r.poisson(gain * 0.03)))
            counts += np.histogram(sp, edges)[0]
        lats.append(latency(counts / 150 / 0.002))
    area_lat[a] = np.nanmedian(lats)
print({a: f"{1000 * v:.0f} ms" for a, v in area_lat.items()})
plt.bar(areas, [1000 * area_lat[a] for a in areas]); plt.ylabel("median latency (ms)"); plt.show()

In [ ]:
import numpy as np
from scipy.stats import spearmanr
def test_latency():
    "latency(rate) finds a response that starts 40 ms after onset to within 5 ms"
    fake = np.where((centres > 0.040) & (centres < 0.08), 60.0, 5.0)
    assert abs(latency(fake) - 0.040) < 0.005

def test_order():
    "the median latencies recover the simulated ordering of areas (Spearman correlation above 0.9)"
    assert spearmanr(lat_mu, [area_lat[a] for a in areas]).correlation > 0.9

test_latency()
test_order()
print("All checks passed")